# Chapter 16: What it costs to ask

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "16"
__file__ = os.path.abspath("run.py")

## The scene

A request from headquarters, two lines long. Every force in the region wants LANTERN's message
classifier: about forty thousand intercepted messages a month between them. What would a year
cost, and would a hosted model, used only on synthetic training material, be cheaper or faster?

Varma passes it to you. "Not a guess," she says. "Measure what one call costs us here, then work
out what it would cost there. And tell me what makes the difference."

The classifier's standing instructions are Okafor's sorting guide,
[`case/sorting-guide.md`](../../case/sorting-guide.md): three labels, eight rules and thirty
worked examples, all invented for training. The messages are the 48 from Reyes's phone,
[`case/messages.jsonl`](../../case/messages.jsonl), with Okafor's labels.

## The concept

A call to a model is paid for in **tokens**. On a hosted model the bill is input tokens times the
input price plus output tokens times the output price; on LANTERN the bill is time, and most of
that time is spent reading the prompt. A classifier sends the same long instructions with every
message, so almost every token of every call is a token it has sent before.

A **prompt cache** stops paying for that again. The model's work on a prompt (the key and value
vectors for each token, the KV cache) depends only on the tokens before it. If a new prompt starts
with exactly the same tokens as an earlier one, that work can be reused, and only the new tail
has to be processed.

![Prefix cache](../../docs/diagrams/16-prefix-cache.svg)

Three ideas follow, and the chapter measures each:

| Idea | What it means | What the run measures |
|---|---|---|
| Cost per call | Tokens in and out, times price; locally, seconds | One call, counted (step 1) |
| Prefix stability | The cache is a prefix match: the first byte that differs ends the reuse | Fixed part first against a request ID first (steps 2 and 3) |
| Cost per correct answer | What you pay divided by what you got right | The estimate in step 5 |

## How it works

**Locally.** Ollama's runner keeps the KV cache of the last prompt it processed. When the next
request starts with the same tokens, it skips them and processes only the rest. Nothing has to be
switched on, and `chat(..., cache=True)` is ignored. Two numbers on every `Reply` show it:

- `prompt_seconds`: Ollama's own prompt processing time (`prompt_eval_duration`);
- `seconds`: the whole call on the wall clock, including model load, prompt processing and
  generation.

**Hosted.** Anthropic caches only when asked. `chat(..., cache=True)` adds a top-level
`cache_control: {"type": "ephemeral"}` to the request (`anthropic_request` in
`src/lantern/cloud.py`): automatic caching of the longest prefix, kept for five minutes and
refreshed by every read. The reply says what happened, and `Reply` carries it:

```python
r = chat([{"role": "system", "content": GUIDE}, {"role": "user", "content": msg}], cache=True)
r.prompt_tokens        # every input token, cached or not
r.cache_write_tokens   # written to the cache on this call (billed at cache_write)
r.cached_tokens        # read from the cache on this call (billed at cache_read)
r.cost_usd             # llm.cost_of(r), from config.yaml `prices:`
```

The prices are in `config.yaml`, copied from the provider's price page and checked on
21 November 2026. For `claude-haiku-5-5`, in USD per million tokens: input 0.10, cache write
0.125, cache read 0.01, output 0.50, and every price five times higher for a request above
100,000 prompt tokens. `llm.cost_of` applies them.

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import statistics
import time
import uuid
from datetime import datetime

from lantern import case, chat, learn
from lantern.config import load_config, model_name, tier
from lantern.llm import Reply, cost_of
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    save_shot,
    step,
    your_turn,
)

chapter("16", "What it costs to ask", """
A request from headquarters, two lines long. Every force in the region wants LANTERN's message
classifier: about forty thousand intercepted messages a month between them. What would a year
cost, and would a hosted model, used only on synthetic training material, be cheaper or faster?

Varma passes it to you. "Not a guess," she says. "Measure what one call costs us here, then work
out what it would cost there. And tell me what makes the difference."
""")

LABELS = ["threat", "logistics", "personal"]
PER_YEAR = 40_000 * 12  # headquarters' figure: about forty thousand messages a month
HOSTED = "claude-haiku-5-5"  # the hosted model the estimate prices (config.yaml `prices:`)

GUIDE = case.read("sorting-guide.md")
messages = case.jsonl("messages.jsonl")
evidence("sorting-guide.md (first lines; the whole guide is the fixed part of every call)",
         "\n".join(GUIDE.splitlines()[:9]))


def ask(m: dict) -> str:
    return f"Message from {m['from']}: \"{m['text']}\""


def stable(m: dict) -> list[dict]:
    """The long fixed part first, the message last: every call starts with the same bytes."""
    return [{"role": "system", "content": GUIDE}, {"role": "user", "content": ask(m)}]


def varying(m: dict) -> list[dict]:
    """The same guide, behind a request ID and a timestamp: no two calls share a first line."""
    header = f"Request ID {uuid.uuid4()}. Received {datetime.now():%Y-%m-%d %H:%M:%S}.\n\n"
    return [{"role": "system", "content": header + GUIDE}, {"role": "user", "content": ask(m)}]


def parse(text: str) -> str:
    for word in text.lower().replace(".", " ").replace(":", " ").split():
        if word in LABELS:
            return word
    return "invalid"


def classify(layout, m: dict) -> Reply:
    return chat(layout(m), temperature=0, max_tokens=8, cache=True)


def hosted_estimate(prompt_tokens: int, output_tokens: int, read: int = 0, write: int = 0):
    """USD on the hosted model for one call of this size, from config.yaml's prices. An
    estimate: Qwen's token counts stand in for Claude's, which a different tokenizer makes."""
    return cost_of(Reply("", HOSTED, provider="anthropic", prompt_tokens=prompt_tokens,
                         output_tokens=output_tokens, cached_tokens=read,
                         cache_write_tokens=write))


local = tier() == "local"
model = model_name(None)

### 1. One call, counted

In [ ]:
step("1. One call, counted")
p_tokens = learn.predict_number(
    f"The sorting guide is {len(GUIDE.split()):,} words. How many prompt tokens will one call "
    "carry, guide and message together?", unit="tokens")
first = classify(stable, messages[0])
# Ollama counts only the prompt tokens it had to process; a call that reuses a cached prefix
# reports fewer. A cold call (a fresh prefix) shows the full size.
cold = classify(varying, messages[0])
full_tokens = cold.prompt_tokens
t = Table(title=f"One call on {model} ({tier()} tier)")
for col in ("", "prompt tokens", "output tokens", "prompt processing s", "whole call s",
            "cost USD"):
    t.add_column(col, justify="left" if not col else "right")
for label, r in (("a fresh prefix", cold), ("the same prefix again", first)):
    t.add_row(label, f"{r.prompt_tokens:,}", str(r.output_tokens),
              f"{r.prompt_seconds:.3f}" if local else "not reported",
              f"{r.seconds:.2f}", f"{r.cost_usd:.6f}" if r.cost_usd is not None else "unknown")
console.print(t)
learn.reveal(p_tokens, actual=full_tokens, tolerance=0.3,
             why="A token is about three quarters of an English word, and the chat template adds "
             "a few more. Every one of them is sent, and on a hosted model billed, on every call.")
note("Locally a call costs nothing in money: it costs time, and the time is mostly the prompt.")
note(f"Ollama's `prompt_eval_count` counts only the tokens it had to process: the same prefix "
     f"again reported {first.prompt_tokens:,}, the fresh one {cold.prompt_tokens:,}." if local
     else "The hosted reply counts every input token, and the cached ones separately.")

### 2. Fixed part first: {len(messages)} messages, the guide always in front

In [ ]:
step(f"2. Fixed part first: {len(messages)} messages, the guide always in front")
p_speed = learn.predict(
    "Ollama keeps the last prompt's KV cache and skips any prefix it has already processed. How "
    "much faster will prompt processing be when only the message changes?",
    ["about the same", "about 2 times", "3 to 20 times", "more than 20 times"])


def run(layout) -> dict:
    start = time.perf_counter()
    replies = [classify(layout, m) for m in messages]
    preds = [parse(r.text) for r in replies]
    return {
        "replies": replies,
        "secs": time.perf_counter() - start,
        "correct": sum(p == m["label"] for p, m in zip(preds, messages, strict=True)),
        "prompt_s": statistics.median(r.prompt_seconds for r in replies),
        "wall_s": statistics.median(r.seconds for r in replies),
        "tokens": statistics.median(r.prompt_tokens for r in replies),
    }


A = run(stable)

### 3. Something that changes, first: the same {len(messages)} calls behind a request ID

In [ ]:
step(f"3. Something that changes, first: the same {len(messages)} calls behind a request ID")
p_hits = learn.predict(
    "Every call now starts with a new request ID and timestamp, and then the same guide. How many "
    f"of the {len(messages)} calls will reuse the cached guide?",
    ["none", "a few", "about half", "nearly all"])
B = run(varying)


COLD_S = statistics.median(r.prompt_seconds for r in B["replies"])  # every one of these is cold


def hit(r: Reply) -> bool:
    """Did this call reuse a cached prefix? Hosted: the provider says so. Local: Ollama
    processed under half the full prompt, or took under a quarter of a cold call's time."""
    if not local:
        return r.cached_tokens > 0
    return r.prompt_tokens < full_tokens / 2 or r.prompt_seconds < COLD_S / 4


for res in (A, B):
    res["hits"] = sum(hit(r) for r in res["replies"])

c = Table(title=f"The same {len(messages)} messages, two layouts ({model}, temperature 0)")
for col in ("layout", "cache hits", "prompt tokens processed (median)",
            "prompt processing s (median)", "whole call s (median)", "run s", "correct"):
    c.add_column(col, justify="left" if col == "layout" else "right")
for name, res in (("fixed part first", A), ("request ID first", B)):
    c.add_row(name, f"{res['hits']}/{len(messages)}", f"{res['tokens']:,.0f}",
              f"{res['prompt_s']:.3f}" if local else "not reported", f"{res['wall_s']:.2f}",
              f"{res['secs']:.0f}", f"{res['correct']}/{len(messages)}")
console.print(c)
speedup = B["prompt_s"] / A["prompt_s"] if A["prompt_s"] else 0.0
wall_speedup = B["wall_s"] / A["wall_s"] if A["wall_s"] else 0.0
learn.reveal(p_speed, answer=learn.bucket(speedup, [1.3, 2.9, 20]),
             why=f"Prompt processing: {speedup:.1f} times faster with the guide cached; the whole "
             f"call {wall_speedup:.1f} times. Only the message's few dozen tokens are new, so only "
             "they are processed. Generation, a few tokens here, does not change.")
learn.reveal(p_hits, answer=learn.bucket(B["hits"], [0, len(messages) // 4,
                                                     3 * len(messages) // 4]),
             why=f"{B['hits']} of {len(messages)}. A cache is a prefix match: the first character "
             "that differs ends the reuse, and the request ID is the first line. The guide behind "
             "it is identical every time and is still processed every time.")
note("Same guide, same messages, same answers: only the order of the bytes changed.")

### 4. What breaks a cache

In [ ]:
step("4. What breaks a cache")


def shared_prefix(a: list[dict], b: list[dict]) -> int:
    """How many characters two requests share from the start (system turns joined)."""
    x = "".join(m["content"] for m in a)
    y = "".join(m["content"] for m in b)
    n = 0
    for ca, cb in zip(x, y, strict=False):
        if ca != cb:
            break
        n += 1
    return n


TOOLS = [
    {"name": "lookup_sender", "description": "Earlier messages from the same sender.",
     "parameters": {"type": "object", "properties": {"sender": {"type": "string"}},
                    "required": ["sender"]}},
    {"name": "lookup_case", "description": "Whether a name appears in an open case.",
     "parameters": {"type": "object", "properties": {"name": {"type": "string"}},
                    "required": ["name"]}},
]
probe = messages[1:7]
tool_runs = {}
for name, order in (("same order", lambda i: TOOLS),
                    ("order swapped each call", lambda i: TOOLS[::1 - 2 * (i % 2)])):
    rs = [chat(stable(m), tools=order(i), temperature=0, max_tokens=8, cache=True)
          for i, m in enumerate(probe)]
    tool_runs[name] = sum(hit(r) for r in rs[1:])  # the first call of each run sets the cache
b = Table(title="Two consecutive requests: where do they stop matching?")
for col in ("change", "characters shared", "of", "cache hits (live)"):
    b.add_column(col, justify="left" if col == "change" else "right")
pair_a = (stable(messages[0]), stable(messages[1]))
pair_b = (varying(messages[0]), varying(messages[1]))
total = len(GUIDE)
b.add_row("fixed part first, next message", f"{shared_prefix(*pair_a):,}", f"{total:,}",
          f"{A['hits']}/{len(messages)}")
b.add_row("request ID and time first", f"{shared_prefix(*pair_b):,}", f"{total:,}",
          f"{B['hits']}/{len(messages)}")
for name, hits in tool_runs.items():
    b.add_row(f"two tools, {name}", "", "", f"{hits}/{len(probe) - 1}")
console.print(b)
swapped, probes = tool_runs["order swapped each call"], len(probe) - 1
if swapped == probes and local:
    note(f"Swapping the two tools kept {swapped} of {probes} hits here: qwen3's chat template puts "
         "the tool list after the system text, so the guide in front stayed cached and only what "
         "followed it changed. Anthropic renders tools before the system prompt, so the same swap "
         "there invalidates the whole prefix. Sort tools by name and build them once, and the "
         "order never matters.")
else:
    note(f"Swapping the two tools kept {swapped} of {probes} hits. Where a provider renders tools "
         "ahead of the system text (Anthropic does), a reordered list changes the first bytes and "
         "nothing after is reused. Sort tools by name and build them once.")
note("Other silent breakers: a date in the system prompt, a dict serialised in a different "
     "order, a per-user line near the top, a changed model or a different thinking setting.")

### 5. A year of messages on a hosted model: an estimate, not a measurement

In [ ]:
step("5. A year of messages on a hosted model: an estimate, not a measurement")
p_year = learn.predict(
    f"With the guide cached, what share of the uncached cost would a year ({PER_YEAR:,} messages) "
    f"cost on {HOSTED}?",
    ["about the same", "about half", "about a quarter", "under a fifth"])
price = load_config()["prices"][HOSTED]
out_tokens = round(statistics.mean(r.output_tokens for r in A["replies"]))
suffix = max(full_tokens - round(statistics.median(r.prompt_tokens for r in A["replies"])), 1)
prefix = full_tokens - suffix
per_plain = hosted_estimate(full_tokens, out_tokens)
per_write = hosted_estimate(full_tokens, out_tokens, write=prefix)
per_read = hosted_estimate(full_tokens, out_tokens, read=prefix)
year_plain = per_plain * PER_YEAR
# Continuous traffic keeps a 5-minute cache warm: price one write per batch of 1,000 calls,
# which is conservative, and reads for the rest.
year_cached = (per_write + 999 * per_read) / 1000 * PER_YEAR
accuracy = A["correct"] / len(messages)
y = Table(title=f"Estimate for {HOSTED}: prices from config.yaml (checked {price['checked']}), "
          "token counts from this run's Qwen tokenizer")
for col in ("", "per call USD", f"per year USD ({PER_YEAR:,} calls)"):
    y.add_column(col, justify="left" if not col else "right")
y.add_row("no cache", f"{per_plain:.7f}", f"{year_plain:,.2f}")
y.add_row("cache write (first call)", f"{per_write:.7f}", "")
y.add_row("cache read (the rest)", f"{per_read:.7f}", "")
y.add_row("with cache", "", f"{year_cached:,.2f}")
y.add_row(f"per correct answer, at this run's {accuracy:.0%}",
          f"{year_cached / PER_YEAR / accuracy:.7f}" if accuracy else "no correct answers", "")
console.print(y)
share = year_cached / year_plain if year_plain else 0.0
learn.reveal(p_year, answer=learn.bucket(share, [0.8, 0.4, 0.2]),
             why=f"{share:.0%} of the uncached cost. The guide is {prefix:,} of the "
             f"{full_tokens:,} prompt tokens, and a cache read costs a tenth of the input price. "
             "What is left is the message, the output and the occasional write.")
under = hosted_estimate(100_000, out_tokens)
over = hosted_estimate(100_001, out_tokens)
note(f"Haiku's price tier: a 100,000-token request costs ${under:.4f}, a 100,001-token one "
     f"${over:.4f}. Packing a whole day's messages into one huge prompt can cost more, not less.")
note("Every number in this table is price arithmetic on token counts, not a hosted run. The "
     "hosted run is `make lesson N=16 TIER=hosted`, which reads cached tokens from the reply.")
if not local:
    spent = sum(r.cost_usd or 0 for r in A["replies"] + B["replies"])
    note(f"This hosted run's measured cost: ${spent:.6f} for {2 * len(messages)} calls.")

learn.record(
    model=model,
    full_prompt_tokens=full_tokens,
    prefix_tokens=prefix,
    calls_per_layout=len(messages),
    cache_hits_stable=A["hits"],
    cache_hits_varying=B["hits"],
    tool_order_hits_same=tool_runs["same order"],
    tool_order_hits_swapped=tool_runs["order swapped each call"],
    median_prompt_s_stable=round(A["prompt_s"], 4),
    median_prompt_s_varying=round(B["prompt_s"], 4),
    median_call_s_stable=round(A["wall_s"], 3),
    median_call_s_varying=round(B["wall_s"], 3),
    prompt_speedup=round(speedup, 1),
    correct_stable=A["correct"],
    correct_varying=B["correct"],
    est_usd_per_call_uncached=round(per_plain, 8),
    est_usd_per_call_cached=round(per_read, 8),
    est_usd_year_uncached=round(year_plain, 2),
    est_usd_year_cached=round(year_cached, 2),
)

your_turn(
    "16",
    check=[
        "In step 1, compare the prompt tokens of the fresh prefix with the same prefix again. "
        "What does Ollama's count leave out, and what would a hosted bill count?",
        "In step 3, the two layouts sent the same guide. Why did only one of them reuse it?",
        "In step 5, which line of the estimate would change if the model's tokenizer were "
        "different, and which would not?",
    ],
    exercises=[
        "In run.py, in stable(), add f\"Today is {datetime.now():%Y-%m-%d %H:%M:%S}.\" to the "
        "start of the system content. How many cache hits are left?",
        "In run.py, in varying(), move the header from the start of the system content to the "
        "start of the user turn. Do the hits come back?",
        "In run.py, cut the worked examples out of the guide you send (GUIDE.split('## Worked "
        "examples')[0]). What happens to the year estimate, and to the correct count?",
        "No edit: run LANTERN_MODEL=qwen3:4b make lesson N=16. Does the speedup grow or shrink "
        "with a larger model?",
    ],
)

finding(f"""
On LANTERN a call costs nothing but time, and most of the time is the prompt. With the sorting
guide first and the message last, {A['hits']} of {len(messages)} calls reused the cached guide and
prompt processing was {speedup:.1f} times faster. A request ID in the first line cut the reuse to
{B['hits']}, with the same guide and the same answers.

The hosted cost is price arithmetic until a hosted run measures it, and the screen says so.

Varma reads the line that says it is an estimate twice. "Then the next thing we prove," she says,
"is that LANTERN can take forty thousand messages without falling over."
""")
save_shot("16-prompt-caching-and-cost")

## The nuances

**1. Locally, the cost is time, and the time is the prompt.** One classification call sends
1,507 prompt tokens and gets 2 back. Of a cold call's 1.62 seconds, 1.42 went on reading the
prompt. A local call costs nothing in money, so the bill you can cut is latency, and the part of
the latency you can cut is the prompt.

**2. A cache is a prefix match.** Ollama keeps the processed tokens of the last prompt and reuses
them up to the first byte that differs. With the guide first and the message last, every one of
48 calls reused the guide; with a request ID and a time in the first line, none did, though the
guide was byte for byte the same in both. Order decides it, not content.

**3. Ollama counts what it processed, not what you sent.** `prompt_eval_count` reported 1,507
tokens for a fresh prefix and 1,450 for the same prefix again: the difference is tokens it did not
have to process. A hosted bill counts every input token and reports the cached ones separately
(`cache_read_input_tokens`), so compare like with like.

**4. What breaks a cache.** Anything early in the prompt that changes between calls: a timestamp
or request ID in the system prompt, a per-user line near the top, a dictionary serialised in a
different order, a tool list built from a set. Where tools go depends on
the template. Anthropic renders tools before the system prompt, so reordering them changes the very
first bytes. qwen3's template puts them after the system text, so in this run swapping the two
tools on every call still kept 5 of 5 hits on the guide in front. Sort tools by name and build them
once, and you never need to know which kind of template you are on. Changing the model
or the thinking setting also starts a new cache: caches belong to one model.

**5. Cost per correct answer, not cost per call.** A cheaper call that is wrong more often is not
cheaper. Divide what a run costs by how many answers were right. Both layouts got 34 of 48 right (71%):
caching changed the speed, not the answers, which is what you want from it.

**6. The long-prompt tier.** Packing many messages into one prompt saves the repeated guide, but
Haiku bills a request above 100,000 prompt tokens at five times the price, for the whole request.
In step 5, a 100,000-token request is priced at $0.0100 and a 100,001-token one at $0.0500. Packing
a day's messages into one prompt can cost more, not less.

**7. An estimate is not a measurement.** Step 5 prices the hosted model with Qwen's token counts.
Claude's tokenizer splits the same text differently, so the numbers are for the order of
magnitude and for comparing layouts. They say "estimate" on the screen and here. The hosted run
reads the real counts, cache reads and writes included, from the reply.

**8. Hosted caches have a floor and a lifetime.** A prefix shorter than the model's minimum
cacheable length is not cached at all, silently: `cache_write_tokens` stays 0. The sorting guide
is well above it. An entry lives five minutes from its last use, so a steady stream of calls
keeps it warm and a batch run once a day pays the write again each time.

## On a hosted model

```bash
make lesson N=16 TIER=hosted            # needs ANTHROPIC_API_KEY in .env
make lesson N=16 TIER=hosted REPLAY=1   # from a recorded run, once one is committed
```

The hosted path uses Anthropic's native prompt caching: `cache=True` sends top-level
`cache_control` (automatic caching of the longest prefix, five-minute lifetime), and the chapter
counts a cache hit from `cache_read_input_tokens` in the reply, not from timing. Claude does not
take `temperature`, `top_k`, `top_p` or `seed`, so the run's `temperature=0` is not sent and is
reported once on the console. Ollama's `prompt_seconds` has no hosted equivalent and shows as
"not reported".

| Measure | Local: qwen3:1.7b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| cache hits, fixed part first (of 48) | 48 | not run |
| cache hits, request ID first (of 48) | 0 | not run |
| cached input tokens per call | not reported by Ollama | not run |
| correct (of 48) | 34 | not run |
| cost per call, USD | 0 (local) | not run |

Live hosted numbers are not run: this lab has no hosted key. Step 5's figures are arithmetic on
documented prices and Qwen token counts, labelled as an estimate; they are not hosted results and
are not entered in this table.

What the hosted run will test: whether the guide caches on Claude as it does on Ollama (it is above
the minimum cacheable length), what the real cached and written token counts and the cost per
correct answer are, and whether the swapped tool order breaks the cache there, as Anthropic's
render order says it should. That a request ID in front of the guide defeats any prefix cache
should hold on both.

## Try this

1. In `stable()`, put `f"Today is {datetime.now():%Y-%m-%d %H:%M:%S}."` at the start of the system
   content. How many cache hits are left?
2. In `varying()`, move the header from the system content to the start of the user turn. Do the
   hits come back, and does the request ID still reach the model?
3. Send only the rules: `GUIDE.split("## Worked examples")[0]`. What happens to the year estimate,
   and to the correct count? Work out the cost per correct answer for both.
4. Run with `LANTERN_MODEL=qwen3:4b`. Does the speedup grow or shrink with a larger model?

## Master it

After this chapter you can count what a call costs, lay out a prompt so its fixed part is reused,
find what breaks a cache before it reaches the bill, and price a run from its usage, labelled as
an estimate when it is one.

**Predict, then see.** `make lesson N=16` stops four times to ask: how many prompt tokens one
call carries, how much faster prompt processing is with the guide cached, how many calls reuse
the cache when a request ID comes first, and what share of the uncached cost a cached year would
be. Each answer is checked against your own run. Add `ARGS=--no-predict` to watch without
stopping.

**Build it.** Write `run_cost()` and `cache_breakers()` in
[`practice/exercise.py`](practice/exercise.py): the cost of a run from call-log records and a
price table, cache reads, writes and the long-prompt tier included, and a check that finds what in
a prompt would break its cache.

```bash
make exercise N=16          # 9 offline tests, no model needed
make hint N=16              # one hint at a time
make solution N=16          # the reference, once you have tried
```

**Check yourself.** `make quiz N=16`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=16` checks your latest run against two targets: a date at the top
of the stable layout, so it misses its cache; and the request ID kept on every call, moved so the
guide is reused again.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 16

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 16

In [ ]:
%run ../../scripts/learn.py hint 16

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 16

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

On LANTERN a call costs nothing but time, and most of the time is the prompt. With the sorting
guide first and the message last, 48 of 48 calls reused the cached guide and prompt processing was
20.3 times faster. A request ID in the first line cut the reuse to 0, with the same guide and the
same answers. The hosted cost is price arithmetic until a hosted run measures it, and the screen
says so.

Varma reads the line that says it is an estimate twice. "Then the next thing we prove," she says,
"is that LANTERN can take forty thousand messages without falling over."

**Next:** [Chapter 17: Forty thousand messages](../17-reliable-calls-at-scale/README.md)